# EDA corregido: aptitud solar fotovoltaica (Entrega 3)

Proyecto final, Machine Learning, Pregrado en Ciencia de Datos.
Integrantes: Jesús David Arévalo Montilla, Enmanuel David Díaz Molinares, Alex David Terán Meza.

Dataset: *Global Dataset of Solar Power Plants* (Mantilla-Guerra et al., 2026, arXiv:2603.20601),
disponible en [`cimejia/solarPV`](https://github.com/cimejia/solarPV).

Este cuaderno retoma la Entrega 1 y corrige los errores críticos de la Entrega 1, tal como pide la guía de la Entrega 3 en su sección 7.1. Antes de
construir los 140 modelos de clasificación y regresión que exige la guía, necesitamos que el target
esté bien definido, que los problemas de calidad de datos estén tratados y no solo contados, y que
el esquema de validación cruzada sea el correcto para este dataset.

El paper fuente define la aptitud solar con la ecuación 1 (página 11):

$$IAS = 0.40\cdot Slope + 0.25\cdot Aspect + 0.20\cdot Hillshade + 0.15\cdot Curvature$$

Las cuatro capas de entrada se clasificaron según FAO (1976) para la pendiente, Wilson y Gallant
(2000) para la curvatura, y se combinaron con ponderación tipo AHP (Saaty, 1980), el mismo método
jerárquico de criterios que usan los propios autores del dataset. Como el índice es totalmente topográfico (no incluye irradiancia ni otras variables climáticas), medirlo con el clima es conceptualmente extraño, pero lo usaremos porque el planteamiento del curso pide un target de clasificación y uno de
regresión, y aquí tenemos ambos en la misma variable: continua para regresión (`solar_aptitude`) y
discretizada en Baja, Media y Alta para clasificación (`solar_aptittude_class`). Solo hay que tener presente que evalúa el terreno. Esa aclaración se desarrolla
con evidencia en la sección 9.

## Pipeline

De los datos crudos a la base limpia que usan los modelos.

![Pipeline](fig_pipeline_eda_mundial.png)

## 1. Librerías y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, StratifiedGroupKFold, cross_val_score
from sklearn.metrics import r2_score

SEED = 42
np.random.seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (8, 4.5)

CLASS_ORDER = ["Baja", "Media", "Alta"]
CLASS_COLORS = {"Baja": "#e76f51", "Media": "#e9c46a", "Alta": "#2a9d8f"}
REGION_ORDER = ["Américas", "Europa/África/M.Oriente", "Asia/Oceanía"]

## 2. Carga de datos

La Entrega 1 documentó la carga con `pandas.read_excel` sobre un archivo `.xlsx`. El archivo real
del proyecto es un CSV con separador `;`, decimal `,` y BOM UTF-8. Con los parámetros por defecto de
pandas, esas tres cosas juntas producen una sola columna de texto en vez de 29 columnas numéricas, así
que el ETL original no era reproducible con los datos que en realidad usamos.

In [ ]:
DATA_PATH = "../Dataset_Mundial_Final(2).csv"

df_raw = pd.read_csv(DATA_PATH, sep=";", decimal=",", encoding="utf-8-sig")
print(f"Filas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}")
df_raw.head(3)

In [ ]:
schema = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "n_nulos": df_raw.isna().sum().values,
    "pct_nulos": (df_raw.isna().mean().values * 100).round(2),
})
schema

Una sola columna tiene nulos de verdad: `area`, con 27.7 % de faltantes. El resto del esquema carga
limpio. Esto ya contradice el diagnóstico de la Entrega 1, que daba por buena toda la tabla sin
reportar este hueco. La razón del faltante se explica en el punto D6 más abajo, junto con por qué no
conviene imputarlo a la ligera.

In [ ]:
na_pct = df_raw.isna().mean().sort_values(ascending=False)
na_pct = na_pct[na_pct > 0]
fig, ax = plt.subplots(figsize=(6, 2))
na_pct.plot(kind="barh", ax=ax, color="#e76f51")
ax.set_xlabel("Proporción de nulos")
ax.set_title("Única columna con faltantes en el CSV crudo")
plt.tight_layout()
plt.show()

## 3. Variable objetivo

`solar_aptitude` va de 0 a 0.942, con media 0.68. Al discretizarla en los umbrales 0.4 y 0.6 definidos
por el paper (sección 2.4.6), la clase Alta queda con el 75 % de las plantas, Media con el 22 % y Baja
con apenas el 3 %. Este desbalance ocurre porque la mayoría de las plantas
del dataset ya están construidas en terrenos que el propio índice califica como favorables, algo
esperable si quien construye una planta solar primero revisa la pendiente del terreno. La sección 9 profundiza en el origen de esta minoría.

In [ ]:
print(df_raw["solar_aptitude"].describe())
print()
dist = df_raw["solar_aptittude_class"].value_counts().reindex(CLASS_ORDER)
dist_pct = (dist / len(df_raw) * 100).round(2)
pd.DataFrame({"n": dist, "%": dist_pct})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(df_raw["solar_aptitude"], bins=50, ax=axes[0], color="#2a9d8f")
axes[0].axvline(0.4, color="crimson", ls="--", lw=1)
axes[0].axvline(0.6, color="crimson", ls="--", lw=1)
axes[0].set_title("solar_aptitude (target de regresión)")
axes[0].set_xlabel("IAS")

sns.barplot(x=dist.index, y=dist.values, ax=axes[1], hue=dist.index,
            palette=CLASS_COLORS, legend=False)
for i, (n, p) in enumerate(zip(dist.values, dist_pct.values)):
    axes[1].text(i, n, f"{n:,}\n({p}%)", ha="center", va="bottom", fontsize=9)
axes[1].set_title("solar_aptittude_class (target de clasificación)")

plt.tight_layout()
plt.savefig("fig_distribucion_target_e3.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Calidad de datos

Encontramos catorce problemas de calidad distintos (D1 a D14), cada uno con una causa identificable en la metodología del paper
(secciones 2.4.3 a 2.4.6). Los numeramos D1 a D14 para poder referenciarlos luego, cuando se decide
qué hacer con cada predictora.

### D1. Filas sin cobertura de DEM

El modelo digital de elevación (DEM) que usa el paper viene organizado por continente y con
resolución de 9 arco-segundos. Fuera de esa cobertura, `slope`, `aspect`, `curvature` y `elevation`
quedan en cero, y como esas cuatro variables alimentan directamente la ecuación 1, el IAS también
sale en cero, lo que la clasifica como Baja sin que eso diga nada real sobre el terreno.

In [ ]:
mask_d1 = (
    (df_raw["solar_aptitude"] == 0)
    & (df_raw["slope"] == 0)
    & (df_raw["aspect"] == 0)
    & (df_raw["curvature"] == 0)
    & (df_raw["elevation"] == 0)
)
d1 = df_raw[mask_d1]
n_baja_total = (df_raw["solar_aptittude_class"] == "Baja").sum()
print(f"Filas sin cobertura de DEM: {len(d1)}")
print(f"Todas con clase 'Baja': {(d1['solar_aptittude_class'] == 'Baja').all()}")
print(f"Proporción de la clase Baja total que representan: {len(d1) / n_baja_total * 100:.1f}%")
d1["country"].value_counts().head(8)

Son casi todas islas pequeñas (Reunión, Mauricio, Cabo Verde) donde el DEM regional no tiene
resolución suficiente. No son plantas con mal terreno: son plantas sin dato de terreno. La decisión
es eliminarlas, no imputarlas.

### D2. Filas sin cobertura de raster GSA

Lo mismo pasa con el raster del Global Solar Atlas que alimenta `ghi` y `pv_potential`. Cuando no hay
cobertura, ambas quedan en cero en vez de en nulo, lo que las hace indistinguibles de un valor real
bajo si no se revisan a mano.

In [ ]:
mask_d2 = (df_raw["ghi"] == 0) & (df_raw["pv_potential"] == 0)
mask_wind0 = df_raw["wind_speed"] == 0
d2 = df_raw[mask_d2]
print(f"Filas con ghi=pv_potential=0: {len(d2)}")
print(d2[["country", "ghi", "humidity", "pv_potential", "wind_speed", "solar_aptitude"]])
print(f"\nFilas adicionales con wind_speed=0: {mask_wind0.sum()}")
df_raw.loc[mask_wind0, ["country", "wind_speed", "solar_aptitude"]]

### D3. `aspect = -1`

Esta es simplemente la convención del paper para marcar terreno plano, donde la orientación no
está definida. Lo confirmamos cruzándolo con la columna categórica y con la pendiente.

In [ ]:
mask_d3 = df_raw["aspect"] == -1
d3 = df_raw[mask_d3]
print(f"aspect = -1: {len(d3)} filas ({len(d3) / len(df_raw) * 100:.2f}%)")
print(f"  Coincide con aspect_type == 'Flat': {(d3['aspect_type'] == 'Flat').mean() * 100:.1f}%")
print(f"  Coincide con slope == 0: {(d3['slope'] == 0).mean() * 100:.1f}%")

Coincide al 100 % con ambas condiciones, así que el centinela es consistente y hay que tratarlo como
una categoría propia, no como un ángulo de 359 grados cualquiera. La sección 5 construye un indicador
binario `is_flat` justo para eso.

### D4. `dist_to_road` con valores que no caben en la metodología

El paper calcula esta distancia con la red vial de Sudamérica (GEOFABRIK) proyectada en "South
America Albers Equal Area", y admite en el texto que fuera de esa cobertura el cálculo se hizo de
forma genérica. Su propio máximo reportado es 6.31 km. En el CSV hay filas muy por encima de eso.

In [ ]:
print(df_raw["dist_to_road"].describe())
print()
for km in (6.31, 20, 100):
    n = (df_raw["dist_to_road"] > km * 1000).sum()
    print(f"  filas con dist_to_road > {km} km: {n}")

mask_d4 = df_raw["dist_to_road"] > 100_000
print("\nPaíses más frecuentes entre las filas con dist_to_road > 100 km:")
df_raw.loc[mask_d4, "country"].value_counts().head(8)

El máximo real es 2,296 km, en Pakistán, un país fuera de Sudamérica. Eso coincide con lo que el paper
mismo advierte: fuera de la región para la que se calibró la red vial, la distancia pierde sentido
físico. Tratamos más de 100 km como error de cálculo y lo convertimos en nulo en la sección 5, en vez
de dejarlo entrar al modelo como si fuera una distancia real.

### D5. `area` incoherente con `capacity`

Una planta solar típica instala entre 40 y 100 vatios por metro cuadrado, con un techo físico
alrededor de 200. Si dividimos `capacity` entre `area`, buena parte del dataset queda muy por encima
de eso.

In [ ]:
# capacity está en MW según GEM. El paper etiqueta la columna como kWp en su Tabla 2, pero esa
# unidad no es compatible con una mediana de varios MW en plantas de miles de m^2 (ver D13).
density_w_m2 = df_raw["capacity"] * 1_000_000 / df_raw["area"]
print("Densidad capacidad/área, asumiendo capacity en MW (físico esperado: 40-200 W/m²):")
print(density_w_m2.describe())
print(f"\nPlantas con densidad > 300 W/m²: {(density_w_m2 > 300).sum()}")
print(f"area máxima registrada: {df_raw['area'].max():,.1f} m²")

El propio procedimiento del paper explica por qué: `area` sale de unir cada coordenada con el
polígono de OpenStreetMap más cercano etiquetado como instalación solar. Si el punto cae sobre un
solo panel o un techo en vez de sobre la planta completa, el área queda diminuta y la densidad se
dispara. Este es un artefacto sistemático del método de unión espacial, así que
`area` no sirve como medida confiable del tamaño físico de una planta.

### D6 y D7. `area` faltante y lo que ese faltante revela

In [ ]:
na_area = df_raw["area"].isna()
print(f"NaN en area: {na_area.sum()} ({na_area.mean() * 100:.2f}%)")
print("\nsize para filas con area faltante:")
print(df_raw.loc[na_area, "size"].value_counts())

print("\nComposición de clase según disponibilidad de area:")
rate = (
    df_raw.assign(area_na=na_area)
    .groupby("area_na")["solar_aptittude_class"]
    .value_counts(normalize=True)
    .unstack()[CLASS_ORDER]
    .round(3) * 100
)
rate

Cuando `area` falta, el 90 % de esas plantas son clase Alta, frente al 70 % cuando sí hay dato. El
faltante no es aleatorio: está correlacionado con la clase. También vemos que la columna `size`
asigna "Small" a las 16,329 filas sin área, mezclando plantas que de verdad son pequeñas con plantas
de las que simplemente no se pudo calcular el área. Eso hace que `size`, tal como viene en el dataset,
no sea una variable confiable por sí sola.

### D8. Duplicados

La Entrega 1 afirmó que no había duplicados. Revisando atributos completos, sin contar el
identificador, el nombre o el código, la afirmación no se sostiene.

In [ ]:
feature_cols = [c for c in df_raw.columns if c not in ("OBJECTID", "code", "plant_name")]

exact_dup = df_raw.duplicated().sum()
attr_dup = df_raw.duplicated(subset=feature_cols, keep=False).sum()
print(f"Duplicados exactos (fila completa, incluye ID): {exact_dup}")
print(f"Duplicados de atributos (sin ID/código/nombre): {attr_dup}")

coord_dup_mask = df_raw.duplicated(subset=["latitude", "longitude"], keep=False)
n_coord_groups = df_raw.loc[coord_dup_mask].groupby(["latitude", "longitude"]).ngroups
print(f"\nFilas con coordenadas repetidas: {coord_dup_mask.sum()} en {n_coord_groups} grupos")

class_per_coord = (
    df_raw.loc[coord_dup_mask]
    .groupby(["latitude", "longitude"])["solar_aptittude_class"]
    .nunique()
)
print(f"Grupos de coordenada con más de una clase distinta: {(class_per_coord > 1).sum()}")

Ningún par de filas es idéntico de principio a fin, por eso el chequeo ingenuo de duplicados exactos
da cero y parece confirmar lo que decía la Entrega 1. Pero al excluir solo las columnas de
identificación, aparecen más de 1,500 filas que comparten todo lo demás, y casi 10,000 filas comparten
coordenada con al menos otra planta. En ocho de esos grupos, la misma coordenada tiene dos clases
distintas, lo que confirma que parte del ruido no viene del terreno sino de cómo se construyó el
dataset.

### D9 y D10. Ruido en el propio target

In [ ]:
for edge in (0.4, 0.6):
    near = df_raw[np.isclose(df_raw["solar_aptitude"], edge, atol=0.0005)]
    print(f"solar_aptitude ≈ {edge}: {len(near)} filas -> {near['solar_aptittude_class'].value_counts().to_dict()}")

expected_rounded = (df_raw["solar_aptitude"] * 10).round().astype(int)
mismatch = (expected_rounded != df_raw["solar_aptitude_rounded"]).sum()
print(f"\nInconsistencias de redondeo en solar_aptitude_rounded: {mismatch} de {len(df_raw):,} filas")

Justo en los umbrales de corte hay filas con el mismo valor de IAS clasificadas en clases distintas,
porque el binning se hizo antes de redondear a tres decimales. Esto es ruido de etiqueta irreducible:
no se puede corregir con limpieza, solo documentar como techo del desempeño que cualquier modelo puede
alcanzar cerca de 0.4 y 0.6. `solar_aptitude_rounded` tiene además 255 inconsistencias de redondeo
propias, así que queda fuera de las predictoras y de los targets.

### D11. `wind_speed` sesgada a la baja

In [ ]:
df_raw["wind_speed"].describe()

El paper calcula esta variable a partir de los componentes zonal y meridional del viento (`u10`,
`v10`), ya promediados mensualmente, antes de obtener la rapidez. Promediar direcciones opuestas
dentro del mismo mes cancela parte de la magnitud real del viento. El resultado corresponde a la magnitud del viento vectorial medio anual, que por construcción
es menor. Hay que leerla con ese matiz, no como un sensor de viento real.

### D12. Redundancia categórica

`slope_type`, `curvature_type`, `aspect_type` y `dt_wind` son clasificaciones derivadas de su
variable numérica correspondiente, con rangos que no se solapan.

In [ ]:
print(df_raw.groupby("slope_type")["slope"].agg(["min", "max", "count"]).sort_values("min"))
print()
print(df_raw.groupby("curvature_type")["curvature"].agg(["min", "max", "count"]).sort_values("min"))

Incluir la versión numérica y la categórica como predictoras separadas no añade información nueva,
solo infla la dimensionalidad y, en modelos lineales, el VIF. Usamos la numérica y descartamos su
versión categórica.

### D13. Los máximos del paper no coinciden con los del CSV

In [ ]:
paper_max = {"area": 153_488, "dist_to_road": 6_309.67, "capacity": 57.5}
for col, pmax in paper_max.items():
    within = (df_raw[col] <= pmax).mean() * 100
    print(f"{col:15s} máx. CSV = {df_raw[col].max():>14,.1f} | máx. paper = {pmax:>10,.1f} | "
          f"% de filas del CSV por debajo del máx. del paper = {within:.1f}%")

En las tres variables, el CSV tiene filas muy por encima del máximo que reporta el paper en su propia
sección 3. La explicación más simple es que las figuras del artículo están filtradas o recortadas
antes de graficarse, mientras que el CSV público no pasó por ese mismo filtro. Al citar los valores
del paper hay que aclarar esa diferencia.

### D14. Clima de baja resolución en algunos países

In [ ]:
for c in ["Argentina", "Colombia", "Chile"]:
    sub = df_raw[df_raw["country"] == c]
    print(f"{c:10s} n={len(sub):4d}  humidity únicos={sub['humidity'].nunique():3d}  "
          f"wind_direction únicos={sub['wind_direction'].nunique():3d}")

Argentina tiene 64 plantas pero solo 4 valores distintos de humedad. El clima viene de promedios
mensuales de ERA5 con resolución espacial gruesa, así que dentro de estos países el "clima" es casi
una constante compartida por decenas de plantas, no una medición local. Conviene tenerlo presente al
interpretar la importancia que SHAP le dé a estas variables en el cuaderno de modelos.

## 5. Construcción del dataset limpio

Eliminamos las filas centinela de D1 y D2, y deduplicamos atributos de D8. El resto de los problemas
(D3, D4, D5 a D7, D12) se resuelven con variables derivadas, no borrando filas, porque en esos casos
la fila sigue teniendo información real en sus demás columnas.

In [ ]:
drop_mask = mask_d1 | mask_d2
df_clean = df_raw.loc[~drop_mask].copy()
df_clean = df_clean.drop_duplicates(subset=feature_cols, keep="first")

print(f"Filas: {len(df_raw):,} -> {len(df_clean):,} tras D1+D2+D8 "
      f"({len(df_raw) - len(df_clean)} eliminadas)")
print()
print(df_clean["solar_aptittude_class"].value_counts())
print((df_clean["solar_aptittude_class"].value_counts(normalize=True) * 100).round(2))

In [ ]:
before = df_raw["solar_aptittude_class"].value_counts(normalize=True).reindex(CLASS_ORDER) * 100
after = df_clean["solar_aptittude_class"].value_counts(normalize=True).reindex(CLASS_ORDER) * 100
comp = pd.DataFrame({"Antes de limpiar": before, "Después de limpiar": after})

fig, ax = plt.subplots(figsize=(7, 4))
comp.plot(kind="bar", ax=ax, color=["#adb5bd", "#2a9d8f"])
ax.set_ylabel("% de filas")
ax.set_title("Proporción de clases antes y después de D1, D2 y D8")
plt.tight_layout()
plt.show()

El cambio es pequeño en términos relativos: la limpieza quita 1,002 filas de 58,978, y Baja pasa de
3.21 % a 3.15 %. Aunque no resuelve el desbalance general, esto evita que un 3.4 % de la clase
minoritaria sea, literalmente, ausencia de datos disfrazada de terreno malo.

In [ ]:
# D3: variables circulares para aspect, con indicador de plano
aspect_valid = df_clean["aspect"].where(df_clean["aspect"] != -1)
aspect_rad = np.deg2rad(aspect_valid)
df_clean["aspect_sin"] = np.sin(aspect_rad).fillna(0.0)
df_clean["aspect_cos"] = np.cos(aspect_rad).fillna(0.0)
df_clean["is_flat"] = (df_clean["aspect"] == -1).astype(int)

# wind_direction es circular por naturaleza, sin centinela que tratar aparte
wind_rad = np.deg2rad(df_clean["wind_direction"])
df_clean["wind_sin"] = np.sin(wind_rad)
df_clean["wind_cos"] = np.cos(wind_rad)

# D4: distancias > 100 km se tratan como error de proyección, no como dato real
df_clean["dist_to_road_clean"] = df_clean["dist_to_road"].where(df_clean["dist_to_road"] <= 100_000)
df_clean["log_dist_to_road"] = np.log1p(df_clean["dist_to_road_clean"])

# Macro-región por longitud, para las secciones 6, 9 y 10
def macro_region(lon):
    if lon < -30:
        return "Américas"
    if lon < 60:
        return "Europa/África/M.Oriente"
    return "Asia/Oceanía"

df_clean["macro_region"] = df_clean["longitude"].apply(macro_region)

print("Variables derivadas: aspect_sin, aspect_cos, is_flat, wind_sin, wind_cos, "
      "log_dist_to_road, macro_region")
df_clean[["aspect", "aspect_sin", "aspect_cos", "is_flat",
          "dist_to_road", "log_dist_to_road"]].describe()

Usar seno y coseno en lugar del ángulo en grados es la forma estándar de tratar variables circulares
en estadística: 1 y 359 grados son casi el mismo punto en la brújula, pero numéricamente están casi
tan separados como es posible (Fisher, 1993). Con el ángulo crudo, un modelo lineal o una prueba de
Kruskal-Wallis los trata como extremos opuestos, que es exactamente el error que corregimos aquí para
`aspect` y `wind_direction`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(df_clean["dist_to_road"], bins=60, ax=axes[0], color="#e76f51")
axes[0].set_title("dist_to_road original")
sns.histplot(df_clean["log_dist_to_road"].dropna(), bins=60, ax=axes[1], color="#2a9d8f")
axes[1].set_title("log1p(dist_to_road), >100 km tratado como error")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(y=np.log1p(df_clean["area"].dropna()), ax=axes[0], color="#e76f51")
axes[0].set_title("log1p(area): outliers de D5")
axes[0].set_ylabel("log1p(area en m²)")
sns.boxplot(y=df_clean["log_dist_to_road"].dropna(), ax=axes[1], color="#e76f51")
axes[1].set_title("log1p(dist_to_road): outliers de D4")
axes[1].set_ylabel("log1p(dist_to_road en m)")
plt.tight_layout()
plt.show()

Incluso después de pasar a escala logarítmica, ambas variables conservan una cola larga de outliers.
Eso es coherente con lo encontrado en D4 y D5: esto refleja una mezcla de mediciones buenas y mediciones con error de cálculo.

## 6. Análisis univariado corregido

### 6.1 Geográficas

Más que ver la latitud y la longitud por separado, interesa ver dónde cae cada clase en el mapa,
porque ahí es donde empieza a notarse el patrón que se investiga a fondo en la sección 9.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for cls in ["Alta", "Media", "Baja"]:
    sub = df_clean[df_clean["solar_aptittude_class"] == cls]
    ax.scatter(sub["longitude"], sub["latitude"], s=3, alpha=0.4,
               c=CLASS_COLORS[cls], label=cls)
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.set_title("Ubicación de cada planta, coloreada por clase de aptitud")
ax.legend(markerscale=4, title="Clase")
plt.tight_layout()
plt.savefig("fig_mapa_clases_e3.png", dpi=150, bbox_inches="tight")
plt.show()

La clase Baja (naranja) no se reparte de forma pareja por el mapa: se concentra casi toda en una
franja alrededor de Europa. Si el IAS midiera solo pendiente, orientación, sombra y curvatura, no
habría razón geográfica para eso. Volvemos sobre este punto con números en la sección 9.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.histplot(df_clean["latitude"], bins=50, ax=axes[0], color="#264653")
axes[0].set_title(f"Latitud (media {df_clean['latitude'].mean():.1f}°)")
sns.histplot(df_clean["longitude"], bins=50, ax=axes[1], color="#264653")
axes[1].set_title(f"Longitud (media {df_clean['longitude'].mean():.1f}°)")
df_clean["country"].value_counts().head(10).plot.barh(ax=axes[2], color="#264653")
axes[2].invert_yaxis()
axes[2].set_title("Países con más plantas")
plt.tight_layout()
plt.show()

### 6.2 Topográficas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.histplot(df_clean["elevation"], bins=50, ax=axes[0, 0], color="#2a9d8f")
axes[0, 0].set_title("Elevación (m)")
sns.histplot(df_clean["slope"], bins=50, ax=axes[0, 1], color="#2a9d8f")
axes[0, 1].set_title("Pendiente (°)")
sns.histplot(df_clean["curvature"], bins=50, ax=axes[1, 0], color="#2a9d8f")
axes[1, 0].set_title("Curvatura")
axes[1, 1].scatter(df_clean["aspect_cos"], df_clean["aspect_sin"], s=2, alpha=0.1, c="#2a9d8f")
axes[1, 1].set_title("Orientación (aspect) en seno y coseno")
axes[1, 1].set_xlabel("cos(aspect)")
axes[1, 1].set_ylabel("sin(aspect)")
plt.tight_layout()
plt.show()

Pendiente y curvatura están casi todas pegadas a cero: la inmensa mayoría de las plantas se construye
en terreno plano, algo lógico porque nivelar terreno inclinado cuesta dinero. El gráfico de
orientación en seno y coseno no tiene forma de anillo perfecto: hay más densidad hacia el sur, que es
la orientación que más sol recibe en el hemisferio norte, donde está la mayoría de las plantas.

### 6.3 Climáticas y logísticas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.histplot(df_clean["ambient_temperature"], bins=50, ax=axes[0, 0], color="#e76f51")
axes[0, 0].set_title("Temperatura ambiente (°C)")
sns.histplot(df_clean["humidity"], bins=50, ax=axes[0, 1], color="#e76f51")
axes[0, 1].set_title("Humedad (%)")
sns.histplot(df_clean["wind_speed"], bins=50, ax=axes[1, 0], color="#e76f51")
axes[1, 0].set_title("Viento vectorial medio (m/s), ver D11")
sns.histplot(df_clean["log_dist_to_road"].dropna(), bins=50, ax=axes[1, 1], color="#e76f51")
axes[1, 1].set_title("log1p(dist_to_road)")
plt.tight_layout()
plt.show()

## 7. Análisis bivariado corregido

La Entrega 1 usó correlación de Pearson sobre variables de cola pesada (`area`, `dist_to_road`) y
circulares (`aspect`, `wind_direction`) sin transformar. Pearson asume relaciones lineales y es
sensible a outliers, justo lo que sobra en esas columnas (Hollander, Wolfe y Chicken, 2014). Aquí
usamos Spearman, que trabaja sobre rangos y no asume linealidad, junto con las versiones seno y
coseno de las variables circulares.

In [ ]:
numeric_cols = [
    "latitude", "longitude", "elevation", "slope", "curvature",
    "aspect_sin", "aspect_cos", "log_dist_to_road", "ambient_temperature",
    "humidity", "wind_speed", "wind_sin", "wind_cos", "ghi", "optimal_tilt",
    "pv_potential", "solar_aptitude",
]
corr = df_clean[numeric_cols].corr(method="spearman")

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="RdBu_r", center=0, square=True, cbar_kws={"label": "Spearman r"})
plt.title("Matriz de correlación de Spearman")
plt.tight_layout()
plt.savefig("fig_correlacion_e3.png", dpi=150, bbox_inches="tight")
plt.show()

corr["solar_aptitude"].drop("solar_aptitude").sort_values(key=abs, ascending=False)

La correlación más fuerte con `solar_aptitude` no es con ninguna de las cuatro variables de la
ecuación 1. Es con `longitude`. Eso ya es una señal de que el índice no se comporta como su propia
fórmula promete, y es el hilo que seguimos en la sección 9.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for region, color in zip(REGION_ORDER, sns.color_palette("viridis", 3)):
    sub = df_clean[(df_clean["macro_region"] == region) & (df_clean["slope"] <= 20)]
    ax.scatter(sub["slope"], sub["solar_aptitude"], s=3, alpha=0.15, color=color, label=region)
ax.set_xlabel("Pendiente (°)")
ax.set_ylabel("solar_aptitude")
ax.set_title("Pendiente contra aptitud, por macro-región")
ax.legend(markerscale=6)
plt.tight_layout()
plt.show()

Dentro de cada región, los puntos sí muestran una tendencia negativa leve: más pendiente, menos
aptitud, que es lo que la física del problema predice. Pero las tres nubes de puntos están a alturas
distintas en el eje vertical. La relación local es real; lo que falta es una relación global que
conecte una región con otra. Eso es justo lo que un desplazamiento entre lotes de procesamiento
produciría.

## 8. Factor de inflación de varianza

El VIF mide cuánta varianza de una variable explica una combinación lineal del resto. Un valor por
encima de 5 suele tomarse como señal de colinealidad relevante, y por encima de 10 como señal fuerte,
aunque ese umbral de 10 es una convención sin base estadística firme y conviene tratarlo como guía, no
como regla dura (O'Brien, 2007).

In [ ]:
vif_cols = [
    "latitude", "longitude", "elevation", "slope", "curvature",
    "aspect_sin", "aspect_cos", "wind_sin", "wind_cos",
    "ambient_temperature", "humidity", "wind_speed", "ghi",
]
vif_data = df_clean[vif_cols].dropna()
X_vif = add_constant(vif_data)
vif_table = pd.DataFrame({
    "variable": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
})
print(f"n = {len(vif_data):,} casos completos")
vif_table

Todas las predictoras quedan muy por debajo de 5, con `const` como la única excepción esperable
porque mide la varianza explicada por el intercepto, no por una variable real. Eso confirma que la
codificación seno y coseno no introdujo colinealidad extra, y que las 13 predictoras numéricas pueden
entrar juntas a un modelo lineal sin que la multicolinealidad distorsione sus coeficientes.

## 9. El IAS es topográfico y tiene un desplazamiento por región

Esto corrige tres afirmaciones de entregas anteriores a la vez: que el IAS mide terreno y clima
juntos, que la clase Baja se concentra en cuatro países europeos, y que su dominancia por longitud se
debe a sesgo de muestreo. Lo mostramos en tres pasos: primero, que el IAS no correlaciona con las
variables que su propia fórmula declara; segundo, que sí correlaciona con la posición geográfica;
tercero, que un modelo entrenado en un continente no predice bien en otro.

In [ ]:
print("Spearman(solar_aptitude, variables de la ecuación 1):")
for col in ["slope", "curvature"]:
    r, p = spearmanr(df_clean["solar_aptitude"], df_clean[col])
    print(f"  {col:10s} r={r:+.3f}  p={p:.2e}")

mask_valid = df_clean["aspect"] != -1
r_a, p_a = spearmanr(df_clean.loc[mask_valid, "solar_aptitude"], df_clean.loc[mask_valid, "aspect"])
print(f"  {'aspect':10s} r={r_a:+.3f}  p={p_a:.2e}  (excluyendo el centinela -1)")

r_lon, p_lon = spearmanr(df_clean["solar_aptitude"], df_clean["longitude"])
r_lat, p_lat = spearmanr(df_clean["solar_aptitude"], df_clean["latitude"])
print(f"\n  {'longitude':10s} r={r_lon:+.3f}  p={p_lon:.2e}")
print(f"  {'latitude':10s} r={r_lat:+.3f}  p={p_lat:.2e}")

La pendiente y la curvatura correlacionan muy poco con el IAS, y la orientación prácticamente nada.
La longitud, en cambio, correlaciona casi diez veces más fuerte que la pendiente. Una variable que no
entra en la fórmula del índice explica más de su variación que las que sí entran.

In [ ]:
region_summary = df_clean.groupby("macro_region").agg(n=("solar_aptitude", "size"),
                                                        media_IAS=("solar_aptitude", "mean"))
region_class = (df_clean.groupby("macro_region")["solar_aptittude_class"]
                .value_counts().unstack()[CLASS_ORDER])
region_summary.join(region_class).reindex(REGION_ORDER)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.boxplot(data=df_clean, x="macro_region", y="solar_aptitude", order=REGION_ORDER,
            hue="macro_region", palette="viridis", legend=False, ax=axes[0])
axes[0].set_title("solar_aptitude por macro-región")
axes[0].tick_params(axis="x", rotation=15)

props = (df_clean.groupby("macro_region")["solar_aptittude_class"]
         .value_counts(normalize=True).unstack()[CLASS_ORDER].reindex(REGION_ORDER))
props.plot(kind="bar", stacked=True, color=[CLASS_COLORS[c] for c in props.columns], ax=axes[1])
axes[1].set_ylabel("Proporción")
axes[1].set_title("Composición de clases por macro-región")
axes[1].legend(title="Clase", bbox_to_anchor=(1.02, 1), loc="upper left")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

Asia y Oceanía tienen una media de IAS casi 0.17 puntos por encima de las otras dos regiones, y casi
no tienen clase Baja. Europa, África y Medio Oriente concentran la gran mayoría de la clase Baja. Esto
ya corrige el error puntual de la Entrega 1, que atribuía Baja solo a cuatro países europeos: en
realidad son siete los que suman la mayoría, y el patrón es regional, no nacional.

In [ ]:
baja_by_country = (df_clean[df_clean["solar_aptittude_class"] == "Baja"]["country"]
                    .value_counts().head(10))
fig, ax = plt.subplots(figsize=(7, 4))
baja_by_country.plot(kind="barh", ax=ax, color="#e76f51")
ax.invert_yaxis()
ax.set_title("Países con más plantas en clase Baja")
plt.tight_layout()
plt.show()

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)


def cv_r2(features):
    sub = df_clean.dropna(subset=features + ["solar_aptitude"])
    X, y = sub[features].to_numpy(), sub["solar_aptitude"].to_numpy()
    scores = cross_val_score(HistGradientBoostingRegressor(random_state=SEED), X, y,
                              cv=kf, scoring="r2", n_jobs=-1)
    return scores.mean(), scores.std()

topo_only = ["slope", "aspect_sin", "aspect_cos", "curvature"]
topo_latlon = topo_only + ["latitude", "longitude"]

m1, s1 = cv_r2(topo_only)
m2, s2 = cv_r2(topo_latlon)
print(f"R² CV, HistGradientBoosting, solo topografía:    {m1:.3f} ± {s1:.3f}")
print(f"R² CV, HistGradientBoosting, + latitud/longitud: {m2:.3f} ± {s2:.3f}")

Con solo las cuatro variables que la ecuación 1 declara, un modelo no lineal apenas explica el 18 % de
la varianza del IAS. Añadir dos coordenadas geográficas, que no aparecen en la fórmula, lo sube a 91
%. La mayor parte de lo que el modelo aprende no viene de la topografía declarada.

In [ ]:
def leave_region_out_r2(features):
    sub = df_clean.dropna(subset=features + ["solar_aptitude", "macro_region"])
    out = {}
    for region in sub["macro_region"].unique():
        train = sub[sub["macro_region"] != region]
        test = sub[sub["macro_region"] == region]
        model = HistGradientBoostingRegressor(random_state=SEED).fit(
            train[features], train["solar_aptitude"])
        pred = model.predict(test[features])
        out[region] = r2_score(test["solar_aptitude"], pred)
    return out

all_numeric = ["latitude", "longitude", "elevation", "slope", "curvature",
               "aspect_sin", "aspect_cos", "ambient_temperature", "humidity", "wind_speed"]

print("R² dejando una macro-región fuera del entrenamiento:")
for region, r2 in leave_region_out_r2(all_numeric).items():
    print(f"  Prueba en {region:28s} R² = {r2:+.3f}")

Un R² negativo significa que el modelo predice peor que simplemente usar el promedio de la región.
Entrenado en dos regiones y probado en la tercera, con diez variables numéricas disponibles, el
modelo pierde por completo. Si el IAS reflejara una relación física estable entre terreno y aptitud,
esa relación debería transferirse entre continentes. No se transfiere.

La explicación más simple, consistente con lo que describe el propio paper en su sección 2.4.3, es que
el DEM se procesó por continente y cada lote se normalizó de manera independiente. El repositorio
oficial del dataset (`cimejia/solarPV`) ofrece evidencia adicional en esa dirección: en una versión
anterior del archivo, publicada antes de incorporar Europa, la clase Baja casi no existía, con apenas
204 filas frente a las 1,892 actuales. La gran mayoría de la clase Baja actual entró en el dataset
junto con el lote europeo. Esto no invalida el dataset para este proyecto, pero sí cambia lo que se
puede afirmar con él: el IAS funciona razonablemente bien para comparar plantas dentro de una misma
región, y no es fiable para comparar entre regiones sin controlar ese efecto de lote. La validación
cruzada tiene que diseñarse teniendo esto en cuenta, que es el tema de la siguiente sección.

## 10. Esquema de validación cruzada espacial

Ninguna de las dos entregas anteriores usó partición espacial, pese a que la primera la declaraba
como palabra clave sin implementarla. Dado el hallazgo de la sección 9, una partición aleatoria deja
que coordenadas casi idénticas caigan una en entrenamiento y otra en prueba, lo que permite
interpolación espacial pura y sobreestima el desempeño real del modelo. Roberts et al. (2017)
documentan este problema en detalle para datos con estructura espacial, temporal o jerárquica, y
recomiendan particionar por bloques en vez de al azar quede como estándar en ese tipo de datos. El
notebook de referencia de los propios autores del dataset sigue la misma lógica, con `GroupKFold`
sobre bloques de 0.5 grados.

In [ ]:
df_clean["block_lat"] = (df_clean["latitude"] // 5) * 5
df_clean["block_lon"] = (df_clean["longitude"] // 5) * 5
df_clean["spatial_block"] = (df_clean["block_lat"].astype(str) + "_" +
                              df_clean["block_lon"].astype(str))

print(f"Bloques espaciales de 5°x5°: {df_clean['spatial_block'].nunique()}")

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
y = df_clean["solar_aptittude_class"]
groups = df_clean["spatial_block"]

for fold, (tr_idx, te_idx) in enumerate(sgkf.split(df_clean, y, groups)):
    n_baja_test = (y.iloc[te_idx] == "Baja").sum()
    print(f"  Fold {fold}: n_test={len(te_idx):6,d}  Baja_test={n_baja_test:4d}  "
          f"({n_baja_test / len(te_idx) * 100:.2f}%)")

Usamos 5 grados por lado en vez de los 0.5 del notebook de referencia porque ahí trabajan sobre un
subconjunto de Sudamérica con 7,000 filas, mientras que aquí el dataset es global y tiene casi 58,000.
Con bloques tan finos como 0.5 grados, el número de grupos distintos sería enorme y muchos quedarían
con una sola planta, lo que rompe la lógica de `GroupKFold`. Con 496 bloques de 5 grados, los cinco
pliegues quedan con un porcentaje de Baja parecido entre sí, entre 2.9 % y 3.8 %, así que ningún
pliegue se queda sin ejemplos de la clase minoritaria. Si en el cuaderno de modelos algún pliegue
resulta demasiado desbalanceado para un modelo específico, la alternativa es reducir el bloque a 2.5
grados o agrupar por país en su lugar.

## 11. Conjunto de predictoras final

La regla de inclusión es simple: una variable entra si está disponible antes de invertir en una
coordenada cualquiera, si no es una copia determinista de otra variable ya incluida, y si no arrastra
un centinela sin tratar. Por eso quedan fuera `area` y `size`, que son posteriores a la construcción de
la planta y traen el problema de D5 a D7; los campos `*_type` y `dt_wind`, que son binnings de D12; y
`capacity` junto con `operational_status`, que solo existen después de decidir invertir.

In [ ]:
predictors_final = [
    "latitude", "longitude", "elevation", "slope", "curvature",
    "aspect_sin", "aspect_cos", "is_flat", "wind_sin", "wind_cos",
    "log_dist_to_road", "ambient_temperature", "humidity", "wind_speed", "ghi",
]
print(f"{len(predictors_final)} predictoras finales: {predictors_final}")

X = df_clean[predictors_final]
y_clf = df_clean["solar_aptittude_class"]
y_reg = df_clean["solar_aptitude"]

print(f"\nX: {X.shape}")
na_counts = X.isna().sum()
print("Nulos por columna:")
print(na_counts[na_counts > 0] if na_counts.sum() else "  (ninguno)")

Quedan 63 valores nulos, todos en `log_dist_to_road`, que vienen de las distancias mayores a 100 km
que tratamos como error en la sección 5. Son menos del 0.11 % de las filas, así que una imputación
simple por mediana dentro de cada pliegue de entrenamiento basta para el cuaderno de modelos, sin
necesidad de descartar esas filas.

In [ ]:
export_cols = predictors_final + [
    "solar_aptitude", "solar_aptitude_rounded", "solar_aptittude_class",
    "spatial_block", "macro_region", "country", "code",
]
df_export = df_clean[export_cols].copy()
df_export.to_csv("dataset_eda_corregido.csv", index=False)
print(f"Exportado: dataset_eda_corregido.csv "
      f"({df_export.shape[0]:,} filas, {df_export.shape[1]} columnas)")

## 12. Conclusiones

El target que vamos a usar en el cuaderno de modelos no cambia respecto al planteado en la Entrega 2:
`solar_aptittude_class` para clasificación y `solar_aptitude` para regresión, la misma variable en dos
formas. Lo que sí cambia es cómo la describimos y cómo la validamos, y eso es gran parte de lo que
este cuaderno corrige.

Sobre calidad de datos, documentamos y tratamos doce problemas que la Entrega 1 no reportó o reportó
de forma incompleta. El más importante para el curso es D8: la afirmación de que el dataset no tenía
duplicados era falsa, con más de 1,500 duplicados de atributos y casi 10,000 filas que comparten
coordenada con alguna otra. D1, D2 y D5 a D7 muestran que varios huecos del dataset no son aleatorios,
sino artefactos del método de construcción: cobertura incompleta de DEM y de GSA, y un procedimiento
de unión punto-polígono que falla sistemáticamente al calcular `area`.

El hallazgo con más peso para todo lo que viene después es el de la sección 9. El IAS correlaciona
casi diez veces más fuerte con la longitud que con la pendiente, pese a que la pendiente pesa 40 % en
la fórmula del índice y la longitud no entra en absoluto. Un modelo entrenado en dos macro-regiones
pierde por completo al predecir la tercera. Esto no significa que el dataset esté mal construido para
los fines de este curso: significa que hay que ser honestos sobre qué mide el índice. Está fuertemente ligado a la macrorregión o lote de procesamiento, con una relación topográfica real pero secundaria
dentro de cada región. Eso convierte la validación cruzada espacial de la sección 10, con bloques de 5
grados y `StratifiedGroupKFold`, en un requisito y no en una opción entre varias.

De cara al cuaderno de modelos, tenemos que tener en cuenta que cualquier métrica
reportada con partición aleatoria va a verse mejor de lo que el modelo realmente generaliza, así que el
bucle externo de la validación anidada tiene que usar `spatial_block` como grupo, nunca una partición
al azar. Además el desbalance de clases (75 % Alta, 22 % Media, 3 % Baja) sigue intacto después de la
limpieza, así que las cuatro técnicas de balanceo que exige la guía, SMOTE (Chawla et al., 2002),
ADASYN (He et al., 2008) y `class_weight='balanced'`, tienen un motivo real detrás y no son solo un
ejercicio obligatorio. Por último al interpretar SHAP sobre el mejor modelo, conviene anticipar que
`latitude` y `longitude` van a dominar la importancia de variables, seguidas de `slope` con una
relación negativa dentro de cada región y de `log_dist_to_road`. Si eso se confirma, hay que
reportarlo como el hallazgo central del proyecto, no esconderlo detrás de una métrica de desempeño
alta.

El siguiente cuaderno, `Experimento_2_Clasificacion.ipynb y los demás cuadernos experimentales`, parte de `dataset_eda_corregido.csv`
y entrena los 112 modelos de clasificación y 28 de regresión que exige la guía de la Entrega 3,
usando `spatial_block` como grupo en el bucle externo de la validación anidada.

## Apéndice: índice de variables

Columnas del dataset original (Tabla 2 del paper fuente), con la decisión que tomamos para cada una
y por qué.

In [ ]:
variable_index = pd.DataFrame([
    ("OBJECTID",              "Identificación", "entero",  "-",        "Identificador interno del SIG",                          "Excluida: identificador, no predictora"),
    ("code",                  "Identificación", "texto",   "-",        "Código alfanumérico de la planta",                       "Excluida: identificador"),
    ("plant_name",            "Identificación", "texto",   "-",        "Nombre de la planta (GEM / GSA)",                        "Excluida: identificador"),
    ("country",                "Geográfica",     "texto",   "-",        "País de la planta",                                      "Excluida como predictora directa; usada como grupo alternativo de CV"),
    ("latitude",               "Geográfica",     "numérica","grados",   "Coordenada de latitud",                                  "Incluida"),
    ("longitude",              "Geográfica",     "numérica","grados",   "Coordenada de longitud",                                 "Incluida: domina la correlación con el target, ver sección 9"),
    ("elevation",              "Topográfica",    "numérica","m",        "Elevación sobre el nivel del mar",                       "Incluida"),
    ("slope",                  "Topográfica",    "numérica","grados",   "Inclinación del terreno",                                "Incluida: componente de la ecuación 1 (peso 0.40)"),
    ("aspect",                 "Topográfica",    "numérica","grados",   "Orientación de la pendiente, -1 si es plana",            "Transformada a aspect_sin, aspect_cos, is_flat (D3)"),
    ("curvature",              "Topográfica",    "numérica","-",        "Forma del terreno, cóncavo o convexo",                   "Incluida: componente de la ecuación 1 (peso 0.15)"),
    ("slope_type",             "Topográfica",    "categórica","-",      "Clasificación de slope",                                 "Excluida: binning determinista de slope (D12)"),
    ("aspect_type",            "Topográfica",    "categórica","-",      "Clasificación de aspect",                                "Excluida: binning determinista de aspect (D12)"),
    ("curvature_type",         "Topográfica",    "categórica","-",      "Clasificación de curvature",                             "Excluida: binning determinista de curvature (D12)"),
    ("dist_to_road",           "Logística",      "numérica","m",        "Distancia a la carretera más cercana",                   "Transformada a log_dist_to_road, >100 km tratado como error (D4)"),
    ("area",                   "Logística",      "numérica","m2",       "Área del polígono de la planta en OSM",                  "Excluida: posterior a la construcción, densidad inconsistente (D5-D7)"),
    ("size",                   "Logística",      "categórica","-",      "Clasificación de area",                                  "Excluida: hereda el problema de area, confunde NaN con 'Small' (D6)"),
    ("ambient_temperature",    "Climática",      "numérica","°C",       "Temperatura media anual",                                "Incluida"),
    ("humidity",               "Climática",      "numérica","%",        "Humedad relativa media anual",                           "Incluida"),
    ("wind_speed",             "Climática",      "numérica","m/s",      "Magnitud del viento vectorial medio",                    "Incluida, con la salvedad de D11"),
    ("wind_direction",         "Climática",      "numérica","grados",   "Dirección predominante del viento",                      "Transformada a wind_sin, wind_cos"),
    ("dt_wind",                "Climática",      "categórica","-",      "Clasificación de wind_direction",                        "Excluida: binning determinista (D12)"),
    ("operational_status",     "Potencia",       "categórica","-",      "Fase del proyecto (operando, cancelado, etc.)",          "Excluida: posterior a la decisión de invertir"),
    ("solar_aptitude",         "Potencia",       "numérica","-",        "Índice de aptitud solar (IAS), ecuación 1",              "Target de regresión"),
    ("solar_aptitude_rounded", "Potencia",       "entero",  "-",        "IAS redondeado a un dígito",                             "Excluida: inconsistencias de redondeo propias (D10)"),
    ("solar_aptittude_class",  "Potencia",       "categórica","-",      "IAS discretizado en Baja, Media, Alta",                  "Target de clasificación"),
    ("optimal_tilt",           "Potencia",       "numérica","grados",   "Ángulo óptimo de inclinación del panel",                 "Disponible antes de invertir; se deja fuera del conjunto base por redundancia con latitud, decisión abierta del equipo"),
    ("ghi",                    "Potencia",       "numérica","kWh/m2/día","Irradiación global en plano inclinado (GTI real)",      "Incluida"),
    ("capacity",               "Potencia",       "numérica","MW",       "Potencia instalada",                                     "Excluida: posterior a la decisión de invertir"),
    ("pv_potential",           "Potencia",       "numérica","kWh/kWp/día","Energía generada por kWp instalado",                   "Disponible antes de invertir; se deja fuera del conjunto base por redundancia con ghi, decisión abierta del equipo"),
], columns=["variable", "categoría", "tipo", "unidad", "descripción", "decisión en este EDA"])

pd.set_option("display.max_colwidth", 90)
variable_index

`optimal_tilt` y `pv_potential` están disponibles antes de construir la planta, así que no violan la
regla de fuga temporal, pero las dejamos fuera del conjunto base porque están muy correlacionadas con
variables ya incluidas (`optimal_tilt` con la latitud, `pv_potential` con `ghi`). Si el cuaderno de
modelos necesita una variante con más variables, esta tabla ya documenta la razón para añadirlas o no.

## Referencias

Mantilla-Guerra, A., Mejia-Escobar, C., Azorin-Lopez, J., Garcia-Rodriguez, J., Tarco, B. F., y
Santamaria, K. (2026). *Global dataset of solar power plants: multidimensional integration and
analysis*. arXiv:2603.20601.

FAO (1976). *A framework for land evaluation*. Food and Agriculture Organization of the United
Nations.

Wilson, J. P., y Gallant, J. C. (2000). *Terrain analysis: principles and applications*. John Wiley
and Sons.

Saaty, T. L. (1980). *The analytic hierarchy process*. McGraw-Hill.

Roberts, D. R., Bahn, V., Ciuti, S., Boyce, M. S., Elith, J., Guillera-Arroita, G., Hauenstein, S.,
Lahoz-Monfort, J. J., Schröder, B., Thuiller, W., Warton, D. I., Wintle, B. A., Hartig, F., y
Dormann, C. F. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or
phylogenetic structure. *Ecography*, 40(8), 913 a 929.

Fisher, N. I. (1993). *Statistical analysis of circular data*. Cambridge University Press.

Hollander, M., Wolfe, D. A., y Chicken, E. (2014). *Nonparametric statistical methods* (3.a ed.).
Wiley.

O'Brien, R. M. (2007). A caution regarding rules of thumb for variance inflation factors. *Quality
and Quantity*, 41, 673 a 690.

Chawla, N. V., Bowyer, K. W., Hall, L. O., y Kegelmeyer, W. P. (2002). SMOTE: synthetic minority
over-sampling technique. *Journal of Artificial Intelligence Research*, 16, 321 a 357.

He, H., Bai, Y., Garcia, E. A., y Li, S. (2008). ADASYN: adaptive synthetic sampling approach for
imbalanced learning. *Proceedings of the IEEE International Joint Conference on Neural Networks*,
1322 a 1328.

King, G., y Zeng, L. (2001). Logistic regression in rare events data. *Political Analysis*, 9(2),
137 a 163.

# Resumen del Proyecto

Este apartado consolida todo el trabajo, desde la recolección de datos hasta el benchmarking y los siete experimentos. Las cifras corresponden a la ejecución de octubre de 2026.

| Cuaderno | Qué hace |
|---|---|
| `EDA_Corregido(Entregable 3).ipynb` | Limpia y explora la base mundial y genera `dataset_eda_corregido.csv`. |
| `Benchmark_Modelos_Base.ipynb` | Corre los modelos base sobre la base mundial. |
| `Clima_a_Generacion_Colombia.ipynb` | Construye la base de generación real de Colombia y prueba modelos preliminares. |
| `Benchmark_Colombia.ipynb` | Corre los modelos base sobre los datos de Colombia. |
| `Experimentos_1_a_7.ipynb` | Detallan el diseño, clasificación, regresión, optimizadores, rendimiento computacional, pruebas estadísticas e interpretabilidad. |
| `Resumen_Final.ipynb` | Cuaderno de síntesis con la ejecución completa y figuras. |

## 1. La pregunta original y cómo evolucionó

Arrancamos queriendo predecir la aptitud de un sitio para albergar una planta solar usando datos climáticos globales. Tomamos el dataset de Mantilla-Guerra et al. (2026), que trae un índice de aptitud solar (`solar_aptitude`). Al revisar la fórmula del paper, nos dimos cuenta de que el índice se calcula solo con variables de terreno (pendiente, orientación, sombreado y curvatura). El clima no influye. Esto nos obligó a dividir el proyecto en dos caminos:
1. Modelar el índice topográfico del dataset global para evaluar su generalización espacial.
2. Construir una base propia con generación real de plantas colombianas para ver cuánto influye realmente el clima.

## 2. Los datos recolectados

Juntamos información de cinco fuentes: el dataset base de Mantilla-Guerra (con potencial solar y datos topográficos), el Global Energy Monitor (para coordenadas y nombres), XM (el operador eléctrico en Colombia, para tener la generación real hora a hora), y Open-Meteo (para la radiación, nubosidad, viento, temperatura y humedad).

El proceso de limpieza fue riguroso. En la base mundial quitamos filas rotas, reparamos más de mil duplicados y resolvimos vacíos que eran sistemáticos. Esto nos dejó con 57,976 plantas y 15 predictoras. Para Colombia, armamos una serie con 8,589 días-planta provenientes de 16 plantas con coordenadas verificadas. 

## 3. El problema oculto del índice global

El hallazgo central del EDA es que el índice casi no correlaciona con la pendiente, a pesar de que esta es el 40% de su fórmula. En cambio, tiene una correlación de +0.58 con la longitud. Vimos que un modelo entrenado en dos continentes tiene un R² negativo cuando predice en un tercer continente. Esto indica que el índice captura principalmente el bloque o lote de procesamiento (por ejemplo, el bloque europeo) en lugar de una regla topográfica universal.

Por esta razón, tuvimos que evaluar todo usando bloques espaciales de 5° × 5°. Partir los datos al azar mezclaba plantas vecinas y daba métricas muy optimistas. 

## 4. Diseño experimental

Con la advertencia espacial en mente, ajustamos todos los hiperparámetros usando validación cruzada anidada: un bucle externo por bloques geográficos, y un bucle interno para buscar hiperparámetros. 

Comparamos modelos lineales (Logística, Ridge, Lasso) contra métodos más flexibles (KNN, SVM, y luego Random Forest y XGBoost). Frente al desbalance de clases (Baja tiene solo el 3%), probamos dar pesos en la función de costo, SMOTE y ADASYN. Evaluamos optimizadores como búsqueda en grilla, búsqueda aleatoria, Optuna, y un algoritmo genético (con una población de 6 individuos). 

## 5. Resultados y métricas clave

En la **base mundial**, Random Forest y XGBoost, junto con el SVM de kernel, logran los mejores resultados (F1 macro alrededor de 0.82 y R² sobre 0.80). Aplastan a los modelos lineales. Pero como vimos en el EDA, parte de este éxito viene de que estos modelos logran identificar la región de la planta. Si quitamos la latitud y longitud, el desempeño cae en picada. 

En la **base de Colombia**, la historia se invierte. El factor de capacidad (FC) es casi proporcional a la radiación que recibe el panel. Por eso, un modelo lineal como Lasso logra explicar un R² de 0.47 del sube y baja diario de la planta, igualando o superando a modelos más complejos que tienden a sobreajustarse a las 6 zonas geográficas disponibles. Vimos empíricamente que cada kWh/m² de radiación suma alrededor de 4.7 puntos al FC.

## 6. Las pruebas estadísticas

Para salir de dudas, aplicamos la prueba de Clark-West y mediciones como la d de Cohen y la prueba de Diebold-Mariano (1995). Los resultados confirmaron sin espacio a duda la superioridad de Random Forest en la base global, con una d de Cohen gigantesca (10.01 respecto a Ridge y Lasso). Clark-West probó de forma contundente que incluir latitud y longitud dispara el poder predictivo (p < 0.001).

## 7. Qué aprendimos sobre los modelos (Interpretabilidad y Rendimiento)

El perfilamiento computacional confirmó que KNN es rápido de entrenar pero muy lento en inferencia. Random Forest y XGBoost concentran su costo al entrenar, logrando evaluar en fracciones de segundo, con picos de memoria bajísimos (el cuello de botella es el procesador, no la RAM). 

Al aplicar técnicas de explicabilidad, notamos que SHAP y LIME ordenan de forma bastante diferente la cola de importancia de variables, pero ambos coinciden en cuáles son los 3 factores más críticos. 

## 8. Conclusiones y lo que queda pendiente

El índice del dataset mundial no mide aptitud desde el clima, sino principalmente la región. Los modelos pueden predecirlo muy bien si "saben" dónde están, pero fracasan en continentes nuevos. Por otro lado, en Colombia confirmamos que la radiación sí explica la mitad del comportamiento diario de una planta real, y un modelo lineal simple basta para capturarlo. 

Queda pendiente medir la tecnología específica de cada planta en Colombia (potencia DC real, tipo de seguidor), lo cual probablemente explicaría la gran diferencia de nivel base que hay entre una planta y otra. Además, trabajar con radiación medida por estaciones en terreno podría romper el techo del R² de 0.50 que nos impone la radiación estimada por satélite.
